# PS26162 / SIH 2026 — Evidence-Based Geographic Partition Selection
This notebook discovers the Train / Validation / Test design from actual Kaggle data. It does not hard-code preferred states. States/UTs are operational containers; ecological, source-class, feature-space and leakage evidence drive selection.

The final notebook writes:
`state_candidate_matrix.csv`, `partition_scores.csv`, `top_5_partitions.csv`, `final_geographic_split.json`, `geographic_selection_report.md`, and plots under `geographic_selection_plots/`.

## Scientific rule
The uploaded benchmark requires all 36 states/UTs to be considered, ecological/ecological-geographic regions to drive scientific partitioning, empirical spatial dependence to determine leakage separation, explicit minority-class adequacy, and separate realistic (A) and adversarial (B) geographic tests. The notebook below implements those requirements as an auditable pipeline.

In [ ]:
!pip -q install geopandas rasterio shapely pyproj requests tqdm scikit-learn scipy h3 pyarrow

In [ ]:
import os, re, json, math, random, warnings, subprocess, zipfile
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from tqdm.auto import tqdm
from shapely.geometry import Point
from shapely.ops import unary_union
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import BallTree
from scipy.stats import wasserstein_distance

import rasterio
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 26162
random.seed(SEED)
np.random.seed(SEED)

WORK = Path("/kaggle/working")
PLOT_DIR = WORK / "geographic_selection_plots"
PLOT_DIR.mkdir(exist_ok=True)
print("Python environment ready.")

## Phase 0 — inspect the actual Kaggle environment before downloads

In [ ]:
def inventory(root, limit=200):
    root = Path(root)
    rows = []
    if not root.exists():
        return pd.DataFrame(columns=["path","size_mb","suffix"])
    for p in root.rglob("*"):
        if p.is_file():
            rows.append({"path": str(p), "size_mb": round(p.stat().st_size/1024**2,3), "suffix": p.suffix.lower()})
            if len(rows) >= limit:
                break
    return pd.DataFrame(rows)

display(inventory("/kaggle/working"))
display(inventory("/kaggle/input"))

p = WORK / "sih2026_h3_daily_features.parquet"
if "df_daily" in globals():
    firms = df_daily.copy()
    source = "df_daily already in memory"
elif p.exists():
    firms = pd.read_parquet(p)
    source = str(p)
else:
    candidates = list(WORK.glob("*.parquet")) + list(Path("/kaggle/input").rglob("*.parquet"))
    if not candidates:
        raise FileNotFoundError("No FIRMS/H3 parquet found. Expected /kaggle/working/sih2026_h3_daily_features.parquet.")
    firms = pd.read_parquet(candidates[0])
    source = str(candidates[0])

print("FIRMS source:", source)
print("Shape:", firms.shape)
print("Columns:", firms.columns.tolist())
display(firms.dtypes.to_frame("dtype"))

date_cols = [c for c in firms.columns if re.search(r"date|time|acq", str(c), re.I)]
h3_cols = [c for c in firms.columns if re.search(r"^h3|h3_", str(c), re.I)]
lat_cols = [c for c in firms.columns if re.fullmatch(r"lat|latitude", str(c), re.I)]
lon_cols = [c for c in firms.columns if re.fullmatch(r"lon|lng|longitude", str(c), re.I)]
label_cols = [c for c in firms.columns if re.search(r"class|label|target|source", str(c), re.I)]
print("Date-like:", date_cols)
print("H3-like:", h3_cols)
print("Lat/Lon:", lat_cols, lon_cols)
print("Label-like:", label_cols)

if date_cols:
    _dt = pd.to_datetime(firms[date_cols[0]], errors="coerce")
    print("Date range:", _dt.min(), "to", _dt.max())

if h3_cols:
    print("Unique H3:", firms[h3_cols[0]].nunique())

## Phase 1 — canonical 36-unit India ADM1 boundary layer

In [ ]:
# geoBoundaries current API: returns machine-readable metadata including a GeoJSON download URL.
api_url = "https://www.geoboundaries.org/api/current/gbOpen/IND/ADM1/"
meta = requests.get(api_url, timeout=60).json()
print("Boundary year:", meta.get("boundaryYearRepresented"))
print("Source:", meta.get("boundarySource"))
print("Declared ADM1 count:", meta.get("admUnitCount"))

india_states = gpd.read_file(meta["gjDownloadURL"]).to_crs(4326)

NAME_COL = next((c for c in ["shapeName","shape_name","NAME_1","name","NAME"] if c in india_states.columns), None)
if NAME_COL is None:
    raise RuntimeError("Could not identify the state/UT name column.")

india_states["state_ut"] = india_states[NAME_COL].astype(str).str.strip()

if len(india_states) != 36:
    raise RuntimeError(f"Expected 36 state/UT geometries, got {len(india_states)}. Stop; do not proceed with an incomplete candidate universe.")

india_states = india_states[india_states.geometry.notna() & ~india_states.geometry.is_empty].copy()
print("Validated state/UT geometries:", len(india_states))
print(india_states["state_ut"].sort_values().tolist())

## Phase 2 — exact FIRMS observation-to-state assignment

In [ ]:
def h3_centroid(cell):
    import h3
    if hasattr(h3, "cell_to_latlng"):
        return h3.cell_to_latlng(cell)
    if hasattr(h3, "h3_to_geo"):
        return h3.h3_to_geo(cell)
    raise RuntimeError("Unsupported h3-py API.")

w = firms.copy()
if lat_cols and lon_cols:
    w["_lat"] = pd.to_numeric(w[lat_cols[0]], errors="coerce")
    w["_lon"] = pd.to_numeric(w[lon_cols[0]], errors="coerce")
else:
    if not h3_cols:
        raise RuntimeError("Need latitude/longitude or H3.")
    hc = h3_cols[0]
    lookup = {}
    for cell in tqdm(w[hc].dropna().astype(str).unique(), desc="H3 centroids"):
        try:
            lookup[cell] = h3_centroid(cell)
        except Exception:
            lookup[cell] = (np.nan, np.nan)
    w["_lat"] = w[hc].astype(str).map(lambda x: lookup.get(x,(np.nan,np.nan))[0])
    w["_lon"] = w[hc].astype(str).map(lambda x: lookup.get(x,(np.nan,np.nan))[1])

w = w[w["_lat"].between(-90,90) & w["_lon"].between(-180,180)].copy()
fg = gpd.GeoDataFrame(w, geometry=gpd.points_from_xy(w["_lon"], w["_lat"]), crs=4326)

joined = gpd.sjoin(
    fg,
    india_states[["state_ut","geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

print("Observations:", len(joined))
print("Assigned to India state/UT:", joined["state_ut"].notna().sum())
print("Unassigned:", joined["state_ut"].isna().sum())

In [ ]:
# State coverage metrics
dcol = date_cols[0] if date_cols else None
hcol = h3_cols[0] if h3_cols else None

area = india_states.to_crs(6933).copy()
area["area_km2"] = area.geometry.area / 1e6

rows = []
for s in india_states["state_ut"]:
    g = joined[joined.state_ut.eq(s)]
    row = {"state_ut":s, "firms_observations":len(g)}
    row["unique_h3"] = g[hcol].nunique() if hcol else np.nan
    row["unique_h3_days"] = g[[hcol,dcol]].drop_duplicates().shape[0] if hcol and dcol else np.nan
    if dcol and len(g):
        dd = pd.to_datetime(g[dcol], errors="coerce")
        row["first_date"] = dd.min()
        row["last_date"] = dd.max()
    else:
        row["first_date"] = pd.NaT; row["last_date"] = pd.NaT
    row["median_obs_per_h3"] = g.groupby(hcol).size().median() if hcol and len(g) else np.nan
    rows.append(row)

state_metrics = pd.DataFrame(rows).merge(area[["state_ut","area_km2"]], on="state_ut")
state_metrics["hotspot_density_per_1000km2"] = state_metrics["firms_observations"] / state_metrics["area_km2"] * 1000
state_metrics["pct_national_obs"] = 100*state_metrics["firms_observations"]/max(1,len(joined))
state_metrics["observed_data_status"] = np.where(state_metrics["firms_observations"]>0,"Observed","Insufficient observed data")
display(state_metrics.sort_values("firms_observations", ascending=False))

## Phase 3 — environmental descriptors: WorldClim + RESOLVE ecoregions + ESA WorldCover

In [ ]:
ENV = WORK / "env"
ENV.mkdir(exist_ok=True)

def dl(url, path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return
    r = requests.get(url, stream=True, timeout=180)
    r.raise_for_status()
    with open(path,"wb") as f:
        for ch in r.iter_content(1024*1024):
            if ch: f.write(ch)

bio_zip = ENV/"wc2.1_10m_bio.zip"
elev_zip = ENV/"wc2.1_10m_elev.zip"
dl("https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_bio.zip", bio_zip)
dl("https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_elev.zip", elev_zip)
for z in [bio_zip,elev_zip]:
    with zipfile.ZipFile(z) as f:
        f.extractall(ENV)

# India sample points for state-level environmental representation
rng = np.random.default_rng(SEED)
N_PER_STATE = 300

def random_points(poly, n):
    minx,miny,maxx,maxy = poly.bounds
    out=[]
    while len(out)<n:
        xs = rng.uniform(minx,maxx,size=n*2)
        ys = rng.uniform(miny,maxy,size=n*2)
        for x,y in zip(xs,ys):
            p=Point(float(x),float(y))
            if poly.contains(p):
                out.append(p)
                if len(out)==n: break
    return out

pts=[]
for _,r in tqdm(india_states.iterrows(), total=len(india_states)):
    for p in random_points(r.geometry,N_PER_STATE):
        pts.append({"state_ut":r.state_ut,"geometry":p})
env_pts = gpd.GeoDataFrame(pts, crs=4326)

# WorldClim BIO1, BIO12, BIO15 + elevation
for num,name in [(1,"bio1_tmean"),(12,"bio12_precip"),(15,"bio15_precip_seasonality")]:
    tif = ENV/f"wc2.1_10m_bio_{num}.tif"
    with rasterio.open(tif) as src:
        env_pts[name] = [v[0] for v in src.sample([(g.x,g.y) for g in env_pts.geometry])]
with rasterio.open(ENV/"wc2.1_10m_elev.tif") as src:
    env_pts["elevation_m"] = [v[0] for v in src.sample([(g.x,g.y) for g in env_pts.geometry])]

# RESOLVE ecoregions, queried only in India's bounding box.
eco_url = "https://data-gis.unep-wcmc.org/server/rest/services/Bio-geographicalRegions/Resolve_Ecoregions/FeatureServer/0/query"
params = {
    "where":"1=1","outFields":"*","returnGeometry":"true","outSR":"4326","f":"geojson",
    "geometry":"68,6,98,36","geometryType":"esriGeometryEnvelope","inSR":"4326",
    "spatialRel":"esriSpatialRelIntersects"
}
eco = gpd.GeoDataFrame.from_features(requests.get(eco_url,params=params,timeout=180).json()["features"],crs=4326)
eco_keep = [c for c in ["eco_id","eco_name","biome_name","realm","geometry"] if c in eco.columns]
env_pts = gpd.sjoin(env_pts, eco[eco_keep], how="left", predicate="within").drop(columns=["index_right"],errors="ignore")

env_summary = env_pts.groupby("state_ut")[[
    "bio1_tmean","bio12_precip","bio15_precip_seasonality","elevation_m"
]].agg(["mean","median","std","min","max"])
env_summary.columns = [f"{a}_{b}" for a,b in env_summary.columns]
env_summary = env_summary.reset_index()
env_summary["ecoregion_count"] = env_pts.groupby("state_ut")["eco_id"].nunique().values
env_summary["biome_count"] = env_pts.groupby("state_ut")["biome_name"].nunique().values

display(env_summary.head())

In [ ]:
# ESA WorldCover 2021: read only COG pixels for the sample points, not the ~117 GB global product.
!aws s3 ls s3://esa-worldcover/v200/2021/map/ --no-sign-request > /kaggle/working/worldcover_tiles.txt

tile_names = []
for line in Path("/kaggle/working/worldcover_tiles.txt").read_text(errors="ignore").splitlines():
    name = line.split()[-1]
    if name.startswith("ESA_WorldCover_10m_2021_v200_N") and name.endswith("_Map.tif"):
        tile_names.append(name)

tile_set=set(tile_names)

def wc_tile(lat,lon):
    la=int(math.floor(lat/3)*3); lo=int(math.floor(lon/3)*3)
    return f"ESA_WorldCover_10m_2021_v200_N{la:02d}E{lo:03d}_Map.tif"

env_pts["_wc_tile"]=[wc_tile(g.y,g.x) if wc_tile(g.y,g.x) in tile_set else None for g in env_pts.geometry]
env_pts["wc_class"]=np.nan

for tile, idx in tqdm(list(env_pts.groupby("_wc_tile").groups.items()), desc="WorldCover COG sampling"):
    if tile is None: continue
    url=f"https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/{tile}"
    try:
        with rasterio.open(url) as src:
            vals=[v[0] for v in src.sample([(env_pts.loc[i].geometry.x, env_pts.loc[i].geometry.y) for i in idx])]
        env_pts.loc[idx,"wc_class"]=vals
    except Exception as e:
        print("WorldCover tile failed:", tile, repr(e))

WC={10:"tree_cover",20:"shrubland",30:"grassland",40:"cropland",50:"built_up",
    60:"bare_sparse",70:"snow_ice",80:"water",90:"wetland",95:"mangrove",100:"moss_lichen"}
env_pts["landcover"]=env_pts["wc_class"].map(WC)

lc=(env_pts.dropna(subset=["landcover"]).assign(n=1)
    .pivot_table(index="state_ut",columns="landcover",values="n",aggfunc="sum",fill_value=0))
lc=lc.div(lc.sum(axis=1),axis=0)
lc.columns=[f"lc_{c}_frac" for c in lc.columns]
lc=lc.reset_index()
env_summary=env_summary.merge(lc,on="state_ut",how="left")
env_summary["forest_sample_frac"]=env_summary.get("lc_tree_cover_frac",np.nan)
env_summary["cropland_sample_frac"]=env_summary.get("lc_cropland_frac",np.nan)
env_summary["builtup_sample_frac"]=env_summary.get("lc_built_up_frac",np.nan)
display(env_summary)

## Phase 4 — actual FIRMS class support and gas-flare candidate support

In [ ]:
# Never invent labels. Prefer an explicit taxonomy column; otherwise only report the known flare proxy.
EXPLICIT_LABEL = None
for c in label_cols:
    u = joined[c].dropna().nunique()
    if 2 <= u <= 20:
        EXPLICIT_LABEL = c
        break

print("Explicit label:", EXPLICIT_LABEL or "DATA UNAVAILABLE")

if {"is_saturated_max","active_days_90d"}.issubset(joined.columns):
    joined["_gas_flare_candidate"] = (
        pd.to_numeric(joined["is_saturated_max"],errors="coerce").eq(1) &
        pd.to_numeric(joined["active_days_90d"],errors="coerce").ge(60)
    )
    gas = joined.groupby("state_ut").agg(
        gas_flare_candidate_h3days=("_gas_flare_candidate","sum")
    )
    if hcol:
        gas["gas_flare_candidate_unique_h3"] = (
            joined.loc[joined["_gas_flare_candidate"]]
            .groupby("state_ut")[hcol].nunique()
        )
    gas["gas_label_status"]="PROXY / CANDIDATE (not ground truth)"
else:
    gas = pd.DataFrame()
    print("Gas flare proxy: DATA UNAVAILABLE")

if EXPLICIT_LABEL:
    observed = (joined.groupby(["state_ut",EXPLICIT_LABEL]).size()
                .rename("observations").reset_index())
    if hcol:
        tmp=(joined.dropna(subset=[EXPLICIT_LABEL])
             .groupby(["state_ut",EXPLICIT_LABEL])[hcol].nunique()
             .rename("unique_h3").reset_index())
        observed=observed.merge(tmp,on=["state_ut",EXPLICIT_LABEL],how="left")
    observed.to_csv(WORK/"firms_observed_label_support.csv",index=False)
    display(observed.sort_values("observations",ascending=False).head(100))
else:
    print("Observed taxonomy class counts: DATA UNAVAILABLE.")

## Phase 5 — WRI coverage

In [ ]:
WRI_URL="https://raw.githubusercontent.com/wri/global-power-plant-database/master/output_database/global_power_plant_database.csv"
try:
    wri=pd.read_csv(WRI_URL)
    print("WRI rows:",len(wri))
    print("WRI note: the repository README says the project is not currently maintained; v1.3.0 is the last release.")
    if {"country","latitude","longitude","primary_fuel"}.issubset(wri.columns):
        wi=wri[wri.country.astype(str).str.upper().eq("IND")].copy()
        wi=wi[pd.to_numeric(wi.latitude,errors="coerce").notna() & pd.to_numeric(wi.longitude,errors="coerce").notna()]
        wg=gpd.GeoDataFrame(wi,geometry=gpd.points_from_xy(wi.longitude,wi.latitude),crs=4326)
        wj=gpd.sjoin(wg,india_states[["state_ut","geometry"]],how="left",predicate="within").drop(columns=["index_right"],errors="ignore")
        wf=wj[wj.primary_fuel.isin(["Gas","Oil","Coal"])].copy()
        wri_support=(wf.groupby(["state_ut","primary_fuel"]).size()
                     .unstack(fill_value=0)
                     .rename(columns={"Gas":"wri_gas","Oil":"wri_oil","Coal":"wri_coal"})
                     .reset_index())
        display(wri_support)
    else:
        raise RuntimeError("Required WRI columns missing.")
except Exception as e:
    wri=None
    wri_support=pd.DataFrame()
    print("WRI: DATA UNAVAILABLE ->",repr(e))

## Phase 6 — OSM coverage

In [ ]:
OSM_PBF=WORK/"india-latest.osm.pbf"
if not OSM_PBF.exists():
    print("Downloading current India OSM PBF (~1.6 GB).")
    subprocess.run([
        "wget","-q","--show-progress",
        "https://download.geofabrik.de/asia/india-latest.osm.pbf",
        "-O",str(OSM_PBF)
    ],check=True)
else:
    print("Using cached OSM:",OSM_PBF)

print("OSM GB:",round(OSM_PBF.stat().st_size/1024**3,3))

STATE_GEO=WORK/"state_geojson"; STATE_OSM=WORK/"state_osm"
STATE_GEO.mkdir(exist_ok=True); STATE_OSM.mkdir(exist_ok=True)

for _,r in india_states.iterrows():
    safe=re.sub(r"[^A-Za-z0-9_]+","_",r.state_ut).strip("_")
    p=STATE_GEO/f"{safe}.geojson"
    if not p.exists():
        gpd.GeoDataFrame([r],crs=4326)[["state_ut","geometry"]].to_file(p,driver="GeoJSON")

def clip_state(state):
    safe=re.sub(r"[^A-Za-z0-9_]+","_",state).strip("_")
    poly=STATE_GEO/f"{safe}.geojson"; out=STATE_OSM/f"{safe}.osm.pbf"
    if out.exists() and out.stat().st_size>0: return state,"cached"
    r=subprocess.run(["osmium","extract","-p",str(poly),str(OSM_PBF),"-o",str(out),"--overwrite"],
                     capture_output=True,text=True)
    if r.returncode!=0: raise RuntimeError(f"{state}: {r.stderr[-1000:]}")
    return state,"clipped"

workers=min(4,os.cpu_count() or 2)
with ThreadPoolExecutor(max_workers=workers) as ex:
    futs=[ex.submit(clip_state,s) for s in india_states.state_ut]
    for f in tqdm(as_completed(futs),total=len(futs),desc="OSM state clips"):
        print(f.result())

OSM_TAGS={
    "industrial":"landuse=industrial",
    "quarry":"landuse=quarry",
    "mineshaft":"man_made=mineshaft",
    "adit":"man_made=adit",
    "farmland":"landuse=farmland",
    "power_plant":"power=plant",
    "power_line":"power=line",
    "road":"highway=*"
}

def fileinfo(txt):
    out={}
    for k in ["nodes","ways","relations"]:
        m=re.search(rf"Number of {k}:\s*([0-9,]+)",txt,re.I)
        out[k]=int(m.group(1).replace(",","")) if m else np.nan
    return out

osm_rows=[]
for s in tqdm(india_states.state_ut,desc="OSM tag counts"):
    safe=re.sub(r"[^A-Za-z0-9_]+","_",s).strip("_")
    state_pbf=STATE_OSM/f"{safe}.osm.pbf"
    row={"state_ut":s}
    for tag_name,tag in OSM_TAGS.items():
        out=STATE_OSM/f"{safe}_{tag_name}.osm.pbf"
        r=subprocess.run(["osmium","tags-filter","-o",str(out),str(state_pbf),f"w/{tag}",f"r/{tag}","--overwrite"],
                         capture_output=True,text=True)
        if r.returncode!=0:
            row[f"osm_{tag_name}_features"]=np.nan
            continue
        info=fileinfo(subprocess.run(["osmium","fileinfo","-e",str(out)],capture_output=True,text=True,check=True).stdout)
        row[f"osm_{tag_name}_features"]=np.nansum([info["ways"],info["relations"]])
    osm_rows.append(row)

osm_support=pd.DataFrame(osm_rows)
display(osm_support.head())

## Phase 7 — assemble the India-wide state candidate matrix

In [ ]:
state_candidate=state_metrics.merge(env_summary,on="state_ut",how="left")
if len(gas): state_candidate=state_candidate.merge(gas.reset_index(),on="state_ut",how="left")
if len(wri_support): state_candidate=state_candidate.merge(wri_support,on="state_ut",how="left")
state_candidate=state_candidate.merge(osm_support,on="state_ut",how="left")

if hcol and "active_days_90d" in joined.columns:
    persistent=(joined.assign(_p=pd.to_numeric(joined.active_days_90d,errors="coerce").ge(30))
                .groupby("state_ut").apply(lambda g:g.loc[g._p,hcol].nunique())
                .rename("persistent_h3_count"))
    state_candidate=state_candidate.merge(persistent,on="state_ut",how="left")
else:
    state_candidate["persistent_h3_count"]=np.nan

state_candidate.to_csv(WORK/"state_candidate_matrix.csv",index=False)
display(state_candidate.sort_values("firms_observations",ascending=False))

## Phase 8 — spatial autocorrelation and empirical buffer

In [ ]:
if hcol:
    activity=None
    for c in ["frp_max","frp_mean","frp","brightness","bright_ti4","active_days_90d"]:
        if c in joined.columns:
            activity=c; break
    agg=joined.groupby(hcol).agg(
        value=(activity,"mean") if activity else (hcol,"size")
    ).reset_index()

    coords=[]; cells=[]
    for cell in tqdm(agg[hcol].astype(str),desc="H3 spatial dependence"):
        try:
            lat,lon=h3_centroid(cell); coords.append((lat,lon)); cells.append(cell)
        except: pass
    z=pd.DataFrame(coords,columns=["lat","lon"]); z[hcol]=cells
    agg=agg.merge(z,on=hcol,how="inner")
    if len(agg)>20000: agg=agg.sample(20000,random_state=SEED).reset_index(drop=True)

    tree=BallTree(np.radians(agg[["lat","lon"]]),metric="haversine")
    k=min(16,len(agg)-1)
    dist,ind=tree.query(np.radians(agg[["lat","lon"]]),k=k+1)
    values=pd.to_numeric(agg["value"],errors="coerce").to_numpy()

    pair=[]
    for i in range(len(agg)):
        for j in range(1,k+1):
            jj=ind[i,j]
            if np.isfinite(values[i]) and np.isfinite(values[jj]):
                pair.append((dist[i,j]*6371.0088,values[i],values[jj]))
    pair=pd.DataFrame(pair,columns=["km","a","b"])

    bins=[0,5,10,20,35,50,75,100,150,200,300,400,600,800,1000]
    corr=[]
    for lo,hi in zip(bins[:-1],bins[1:]):
        p=pair[(pair.km>=lo)&(pair.km<hi)]
        r=np.corrcoef(p.a,p.b)[0,1] if len(p)>=50 and p.a.nunique()>1 and p.b.nunique()>1 else np.nan
        corr.append({"lo_km":lo,"hi_km":hi,"n_pairs":len(p),"correlation":r})
    correlogram=pd.DataFrame(corr)
    empirical_buffer=np.nan
    v=correlogram.dropna(subset=["correlation"]).reset_index(drop=True)
    for i in range(len(v)-1):
        if abs(v.loc[i,"correlation"])<0.1 and abs(v.loc[i+1,"correlation"])<0.1:
            empirical_buffer=float(v.loc[i,"hi_km"]); break
    if not np.isfinite(empirical_buffer) and len(v): empirical_buffer=float(v.hi_km.iloc[-1])

    print("Empirical spatial-dependence buffer:",empirical_buffer,"km")
    display(correlogram)
    correlogram.to_csv(WORK/"spatial_correlogram.csv",index=False)

    plt.figure(figsize=(8,5))
    plt.plot(correlogram.hi_km,correlogram.correlation,marker="o")
    plt.axhline(0,linewidth=1)
    if np.isfinite(empirical_buffer): plt.axvline(empirical_buffer,linestyle="--")
    plt.xlabel("Distance (km)"); plt.ylabel("Neighbor correlation")
    plt.title("FIRMS H3 spatial correlogram")
    plt.tight_layout(); plt.savefig(PLOT_DIR/"spatial_correlogram.png",dpi=160); plt.show()
else:
    empirical_buffer=np.nan
    print("H3 unavailable: spatial autocorrelation NOT COMPUTED.")

## Phase 9 — persistent-source clusters and cross-border ecological systems

In [ ]:
if hcol and "active_days_90d" in joined.columns:
    ph=(joined.assign(active90=pd.to_numeric(joined.active_days_90d,errors="coerce"))
        .groupby(hcol).agg(active90=("active90","max")).reset_index())
    ph=ph[ph.active90>=30]
    cc=[]
    for cell in tqdm(ph[hcol].astype(str),desc="Persistent H3 clusters"):
        try:
            lat,lon=h3_centroid(cell); cc.append((cell,lat,lon))
        except: pass
    pc=pd.DataFrame(cc,columns=[hcol,"lat","lon"])
    pc=pc.merge(ph,on=hcol,how="left")
    from sklearn.cluster import DBSCAN
    eps=(empirical_buffer if np.isfinite(empirical_buffer) else 50)/6371.0088
    pc["cluster_id"]=DBSCAN(eps=eps,min_samples=3,metric="haversine",n_jobs=-1).fit_predict(np.radians(pc[["lat","lon"]]))
    clusters=(pc[pc.cluster_id>=0].groupby("cluster_id")
              .agg(h3_cells=(hcol,"nunique"),max_active_days=("active90","max"),
                   mean_active_days=("active90","mean"),centroid_lat=("lat","mean"),centroid_lon=("lon","mean"))
              .reset_index())
else:
    clusters=pd.DataFrame()
    print("Persistent clusters NOT COMPUTED.")

state_eco_sets=(env_pts.dropna(subset=["eco_id"]).groupby("state_ut")["eco_id"]
                .apply(lambda s:set(s.astype(str))).to_dict())
cross=[]
states=india_states.state_ut.tolist()
for i,a in enumerate(states):
    for b in states[i+1:]:
        sa=state_eco_sets.get(a,set()); sb=state_eco_sets.get(b,set())
        inter=sa&sb; union=sa|sb
        if inter:
            cross.append({"state_a":a,"state_b":b,"shared_ecoregions":len(inter),"jaccard":len(inter)/len(union)})
cross_eco=pd.DataFrame(cross)
cross_eco.to_csv(WORK/"cross_border_ecological_systems.csv",index=False)
display(cross_eco.sort_values("jaccard",ascending=False).head(50))
if len(clusters): clusters.to_csv(WORK/"persistent_source_clusters.csv",index=False)

## Phase 10 — feature-space representation and candidate partitions

In [ ]:
exclude=[r"^h3",r"id$",r"_id$",r"label",r"class",r"target",r"state_ut",r"geometry",r"latitude",r"longitude",r"_lat",r"_lon"]
num=joined.select_dtypes(include=np.number).columns.tolist()
feature_cols=[c for c in num if not any(re.search(p,str(c).lower()) for p in exclude) and joined[c].nunique(dropna=True)>1]

fs=joined[["state_ut"]+feature_cols].copy()
if len(fs)>100000: fs=fs.sample(100000,random_state=SEED)
print("Feature columns:",feature_cols)

# State environmental representation
env_cols=[c for c in [
    "forest_sample_frac","cropland_sample_frac","builtup_sample_frac",
    "bio1_tmean_mean","bio12_precip_mean","bio15_precip_seasonality_mean",
    "elevation_m_mean","ecoregion_count","biome_count",
    "firms_observations","unique_h3","hotspot_density_per_1000km2"
] if c in state_candidate.columns]

E=state_candidate.set_index("state_ut")[env_cols].replace([np.inf,-np.inf],np.nan)
E=E.fillna(E.median(numeric_only=True))
E_std=pd.DataFrame(StandardScaler().fit_transform(E),index=E.index,columns=E.columns)

def group_centroid(states):
    a=[s for s in states if s in E_std.index]
    return E_std.loc[a].mean(axis=0).to_numpy() if a else np.full(E_std.shape[1],np.nan)

def eco_distance(train,test):
    a=group_centroid(train); b=group_centroid(test)
    if np.any(~np.isfinite(a)) or np.any(~np.isfinite(b)): return np.nan,np.nan
    dist=float(np.linalg.norm(a-b))
    near=min(float(np.linalg.norm(group_centroid([s])-b)) for s in train if s in E_std.index)
    return dist,near

def ecoregion_overlap(train,test):
    a=set().union(*(state_eco_sets.get(s,set()) for s in train))
    b=set().union(*(state_eco_sets.get(s,set()) for s in test))
    return len(a&b)/len(b) if b else np.nan

states_all=state_candidate.state_ut.tolist()
candidate_test_sets=[]

for k in [3,4,5,6]:
    lab=KMeans(n_clusters=k,n_init=20,random_state=SEED).fit_predict(StandardScaler().fit_transform(E))
    for cl in np.unique(lab):
        ss=state_candidate.loc[lab==cl,"state_ut"].tolist()
        if 1<=len(ss)<=8: candidate_test_sets.append(("environment_cluster",k,int(cl),tuple(sorted(ss))))

centroids=india_states.to_crs(6933).representative_point().to_crs(4326)
xy=np.column_stack([centroids.x.to_numpy()*np.cos(np.radians(centroids.y.to_numpy())),centroids.y.to_numpy()])
for k in [3,4,5,6,7,8]:
    lab=KMeans(n_clusters=k,n_init=20,random_state=SEED).fit_predict(xy)
    for cl in np.unique(lab):
        ss=india_states.loc[lab==cl,"state_ut"].tolist()
        if 1<=len(ss)<=8: candidate_test_sets.append(("spatial_cluster",k,int(cl),tuple(sorted(ss))))

for size in range(1,9):
    for rep in range(150):
        candidate_test_sets.append(("random",size,rep,tuple(sorted(random.sample(states_all,size)))))

seen=set(); test_sets=[]
for x in candidate_test_sets:
    if x[3] not in seen:
        seen.add(x[3]); test_sets.append(x)
print("Unique candidate test geographies:",len(test_sets))

In [ ]:
# Candidate partitions: varied training sizes, hard support floor.
MIN_TRAIN_OBS=50
MIN_TEST_OBS=30

rows=[]
for family,size,rep,test_tup in test_sets:
    test=list(test_tup)
    remaining=[s for s in states_all if s not in test]
    for tr_size in [2,3,4,5,6,8,10]:
        if tr_size>len(remaining): continue
        for rr in range(3):
            train=random.sample(remaining,tr_size)
            tr_obs=int(state_candidate.loc[state_candidate.state_ut.isin(train),"firms_observations"].sum())
            te_obs=int(state_candidate.loc[state_candidate.state_ut.isin(test),"firms_observations"].sum())
            ed,near=eco_distance(train,test)
            rows.append({
                "candidate_family":family,
                "train_states":json.dumps(sorted(train)),
                "test_states":json.dumps(sorted(test)),
                "train_size":tr_size,"test_size":len(test),
                "train_obs":tr_obs,"test_obs":te_obs,
                "ecological_centroid_distance":ed,
                "nearest_train_environment_distance":near,
                "ecoregion_overlap":ecoregion_overlap(train,test),
                "test_deployment_share":float(state_candidate.set_index("state_ut").loc[test,"firms_observations"].sum()/max(1,len(joined)))
            })
part=pd.DataFrame(rows)

# Feature-space shift: Wasserstein + mean standardized difference
numeric_fs=[c for c in feature_cols if pd.api.types.is_numeric_dtype(fs[c])]
def fs_shift(train,test):
    a=fs[fs.state_ut.isin(train)][numeric_fs]
    b=fs[fs.state_ut.isin(test)][numeric_fs]
    if len(a)==0 or len(b)==0:return np.nan,np.nan
    a=a.sample(min(20000,len(a)),random_state=SEED); b=b.sample(min(20000,len(b)),random_state=SEED)
    ws=[]; smd=[]
    for c in numeric_fs:
        x=pd.to_numeric(a[c],errors="coerce").dropna().to_numpy()
        y=pd.to_numeric(b[c],errors="coerce").dropna().to_numpy()
        if len(x)>=10 and len(y)>=10:
            ws.append(wasserstein_distance(x,y))
            pooled=np.sqrt((np.var(x)+np.var(y))/2)
            smd.append(abs(np.mean(x)-np.mean(y))/pooled if pooled>0 else 0)
    return (np.mean(ws) if ws else np.nan),(np.mean(smd) if smd else np.nan)

part=part.sort_values("ecological_centroid_distance",ascending=False).head(min(1200,len(part))).copy()
sh=[]
for idx,r in tqdm(part.iterrows(),total=len(part),desc="Feature-space shift"):
    tr=json.loads(r.train_states); te=json.loads(r.test_states)
    w,s=fs_shift(tr,te); sh.append((idx,w,s))
for idx,w,s in sh:
    part.loc[idx,"wasserstein_shift"]=w; part.loc[idx,"mean_abs_smd"]=s

## Phase 11 — hard constraints, multi-objective ranking, sensitivity

In [ ]:
def rankpct(s,ascending=True):
    return s.rank(pct=True,ascending=ascending)

# Support rule
part["support_ok"]=(part.train_obs>=MIN_TRAIN_OBS)&(part.test_obs>=MIN_TEST_OBS)

# Preliminary spatial separation proxy at state-centroid scale.
centroids_dict={r.state_ut:(r.geometry.y,r.geometry.x) for _,r in centroids.to_frame(name="geometry").join(india_states.set_index("state_ut"),how="right").reset_index().iterrows()} if False else {}
# Simpler, use projected representative point coordinates:
state_xy=india_states.copy()
state_xy["pt"]=state_xy.to_crs(6933).representative_point().to_crs(4326)
xy_map={r.state_ut:(r.pt.y,r.pt.x) for _,r in state_xy.iterrows()}

def min_state_dist(tr,te):
    a=np.radians([xy_map[s] for s in tr]); b=np.radians([xy_map[s] for s in te])
    return float(BallTree(a,metric="haversine").query(b,k=1)[0].min()*6371.0088)

part["min_state_centroid_km"]=[min_state_dist(json.loads(r.train_states),json.loads(r.test_states)) for r in part.itertuples()]

part["eco_novelty"]=rankpct(part.ecological_centroid_distance)
part["climate_novelty"]=part["eco_novelty"]
part["landcover_novelty"]=rankpct(1-part.ecoregion_overlap.fillna(0))
part["source_class_coverage"]=rankpct(part.test_obs)
part["feature_shift_score"]=rankpct(part.mean_abs_smd)
part["spatial_separation_score"]=rankpct(part.min_state_centroid_km)
part["training_diversity_score"]=rankpct(
    part.train_states.map(lambda x: len(set().union(*(state_eco_sets.get(s,set()) for s in json.loads(x)))))
)
part["deployment_representativeness"]=(1-abs(part.test_deployment_share-0.20)).clip(0,1)

# Preliminary hard rejection. Exact H3/buffer is done on finalists.
part["hard_reject"]=~part.support_ok
if np.isfinite(empirical_buffer):
    part["hard_reject"]|=part.min_state_centroid_km < empirical_buffer*0.25
part["rejection_reason"]=np.where(part.hard_reject,"Insufficient support or preliminary spatial separation","")

survive=part[~part.hard_reject].copy()

criteria=["eco_novelty","climate_novelty","landcover_novelty","source_class_coverage","spatial_separation_score","feature_shift_score","training_diversity_score","deployment_representativeness"]
survive[criteria]=survive[criteria].fillna(survive[criteria].median())

survive["score_equal"]=survive[criteria].mean(axis=1)

weights_general={"eco_novelty":.18,"climate_novelty":.12,"landcover_novelty":.10,"source_class_coverage":.12,"spatial_separation_score":.14,"feature_shift_score":.14,"training_diversity_score":.08,"deployment_representativeness":.12}
survive["score_generalization"]=sum(survive[k]*v for k,v in weights_general.items())

weights_deploy={"eco_novelty":.10,"climate_novelty":.08,"landcover_novelty":.07,"source_class_coverage":.15,"spatial_separation_score":.12,"feature_shift_score":.10,"training_diversity_score":.13,"deployment_representativeness":.25}
survive["score_deployment"]=sum(survive[k]*v for k,v in weights_deploy.items())

def pareto(df,cols):
    A=df[cols].to_numpy()
    keep=np.ones(len(A),bool)
    for i in range(len(A)):
        dom=(np.all(A>=A[i],axis=1)&np.any(A>A[i],axis=1)); dom[i]=False
        if dom.any(): keep[i]=False
    return keep
survive["pareto_optimal"]=pareto(survive,criteria)
print("Preliminary survivors:",len(survive))

## Phase 12 — exact H3 leakage/buffer checks on finalists

In [ ]:
# Build H3 centroid layer once
if hcol:
    uc=joined[[hcol,"state_ut"]].drop_duplicates(hcol)
    pts=[]
    for cell in tqdm(uc[hcol].astype(str),desc="H3 geometry"):
        try:
            lat,lon=h3_centroid(cell); pts.append((cell,lon,lat))
        except: pass
    hp=pd.DataFrame(pts,columns=[hcol,"lon","lat"])
    h3g=gpd.GeoDataFrame(hp,geometry=gpd.points_from_xy(hp.lon,hp.lat),crs=4326).merge(uc,on=hcol,how="left")
else:
    h3g=None

finalists=(survive.sort_values(["pareto_optimal","score_generalization","score_equal"],ascending=[False,False,False])
           .head(250).copy())

def exact_leak(train,test):
    if h3g is None:return {"exact_overlap":np.nan,"buffer_violation":np.nan,"leakage_ok":False}
    test_geom=unary_union(india_states[india_states.state_ut.isin(test)].geometry.tolist())
    test_mask=h3g.geometry.within(test_geom)
    if np.isfinite(empirical_buffer):
        buf=gpd.GeoSeries([test_geom],crs=4326).to_crs(6933).buffer(empirical_buffer*1000).to_crs(4326).iloc[0]
        train_mask=h3g.state_ut.isin(train)&~h3g.geometry.within(buf)
        violation=int((h3g.state_ut.isin(train)&h3g.geometry.within(buf)).sum())
    else:
        train_mask=h3g.state_ut.isin(train); violation=np.nan
    tc=set(h3g.loc[test_mask,hcol]); trc=set(h3g.loc[train_mask,hcol])
    overlap=len(tc&trc)
    return {"exact_overlap":overlap,"buffer_violation":violation,"leakage_ok":(overlap==0 and violation==0)}

for idx,r in tqdm(finalists.iterrows(),total=len(finalists),desc="Exact finalist leakage"):
    x=exact_leak(json.loads(r.train_states),json.loads(r.test_states))
    for k,v in x.items(): finalists.loc[idx,k]=v

finalists["hard_reject_exact"]=~finalists.leakage_ok.fillna(False)
clean=finalists[~finalists.hard_reject_exact].copy()
print("Exact leakage-clean finalists:",len(clean))

## Phase 13 — six sensitivity scenarios and separate Experiment A/B objectives

In [ ]:
SCENARIOS={
    "equal":dict(zip(criteria,[1]*len(criteria))),
    "generalization_focused":{"eco_novelty":2,"climate_novelty":1.5,"landcover_novelty":1.3,"source_class_coverage":1,"spatial_separation_score":1.5,"feature_shift_score":1.5,"training_diversity_score":1,"deployment_representativeness":1},
    "ecology_focused":{"eco_novelty":2.5,"climate_novelty":1.8,"landcover_novelty":1.8,"source_class_coverage":1,"spatial_separation_score":1,"feature_shift_score":1,"training_diversity_score":1.5,"deployment_representativeness":.8},
    "class_support_focused":{"eco_novelty":.8,"climate_novelty":.8,"landcover_novelty":.8,"source_class_coverage":2.5,"spatial_separation_score":1,"feature_shift_score":1,"training_diversity_score":1.5,"deployment_representativeness":1.2},
    "deployment_focused":{"eco_novelty":.8,"climate_novelty":.8,"landcover_novelty":.8,"source_class_coverage":1.2,"spatial_separation_score":1,"feature_shift_score":.8,"training_diversity_score":1.2,"deployment_representativeness":2.5},
    "leakage_averse":{"eco_novelty":1,"climate_novelty":1,"landcover_novelty":1,"source_class_coverage":1,"spatial_separation_score":3,"feature_shift_score":1,"training_diversity_score":1,"deployment_representativeness":1}
}

for name,w in SCENARIOS.items():
    clean[f"score_{name}"]=sum(clean[k]*v for k,v in w.items())/sum(w.values())
    clean[f"rank_{name}"]=clean[f"score_{name}"].rank(ascending=False,method="min")
clean["mean_rank"]=clean[[f"rank_{s}" for s in SCENARIOS]].mean(axis=1)

# Experiment A: realistic unseen deployment geography.
clean["A_score"]=(
    .30*clean.deployment_representativeness+
    .20*clean.source_class_coverage+
    .15*clean.eco_novelty+
    .10*clean.climate_novelty+
    .10*clean.landcover_novelty+
    .10*clean.spatial_separation_score+
    .05*clean.training_diversity_score
)

# Experiment B: adversarial novelty, constrained by the hard support/leakage screen already applied.
clean["B_score"]=(
    .28*clean.eco_novelty+
    .18*clean.climate_novelty+
    .15*clean.landcover_novelty+
    .18*clean.feature_shift_score+
    .10*clean.spatial_separation_score+
    .06*clean.training_diversity_score+
    .05*clean.source_class_coverage
)

best_a=clean.sort_values("A_score",ascending=False).iloc[0]
best_b=clean.sort_values("B_score",ascending=False).iloc[0]
print("Best A:",best_a[["train_states","test_states","A_score"]].to_dict())
print("Best B:",best_b[["train_states","test_states","B_score"]].to_dict())
display(clean.sort_values("mean_rank").head(15)[["train_states","test_states","mean_rank"]+[f"score_{s}" for s in SCENARIOS]])

## Phase 14 — forest novelty verdict

In [ ]:
forest_vars=[c for c in ["forest_sample_frac","bio1_tmean_mean","bio12_precip_mean","bio15_precip_seasonality_mean","elevation_m_mean"] if c in state_candidate.columns]

def forest_test(train,test):
    tr=state_candidate[state_candidate.state_ut.isin(train)][forest_vars].median(numeric_only=True)
    te=state_candidate[state_candidate.state_ut.isin(test)][forest_vars].median(numeric_only=True)
    vec=pd.DataFrame([tr,te],index=["train","test"]).fillna(pd.Series(tr))
    if len(forest_vars):
        zz=StandardScaler().fit_transform(vec)
        d=float(np.linalg.norm(zz[1]-zz[0]))
        verdict="Novel" if d>=2 else ("Partially novel" if d>=1 else "Seen")
    else:
        d=np.nan; verdict="NOT COMPUTED"
    return d,verdict

for name,row in [("A",best_a),("B",best_b)]:
    d,v=forest_test(json.loads(row.train_states),json.loads(row.test_states))
    print(f"Experiment {name}: forest environmental distance={d}, verdict={v}")

## Phase 15 — H3 and temporal leakage rules

In [ ]:
print("H3 recommendation:")
if h3g is not None and np.isfinite(empirical_buffer):
    print("Prefer generalized spatial features over raw H3 IDs. Raw H3 may be retained only with exact H3 exclusion, buffer isolation, and parent/neighbor leakage checks.")
else:
    print("H3 recommendation: NOT COMPUTED because required geometry/buffer evidence is incomplete.")

ta=[]
for c in firms.columns:
    s=str(c).lower()
    if any(k in s for k in ["active_days_90d","persistence_90d","rolling","historical","target_encoding"]):
        status="TARGET LEAKAGE" if "target" in s or "encoding" in s else "SAFE ONLY WITH CAUSAL WINDOW / FOLD-LOCAL RECOMPUTATION"
        ta.append((c,status))
temporal_audit=pd.DataFrame(ta,columns=["feature","status"])
display(temporal_audit)
temporal_audit.to_csv(WORK/"temporal_leakage_audit.csv",index=False)

## Phase 16 — top five partitions and reproducible outputs

In [ ]:
top5=clean.sort_values(["mean_rank","A_score"],ascending=[True,False]).head(5).copy()
top5.to_csv(WORK/"top_5_partitions.csv",index=False)
part.to_csv(WORK/"partition_scores.csv",index=False)

final_train=json.loads(best_a.train_states)
final_a=json.loads(best_a.test_states)
final_b=json.loads(best_b.test_states)

design={
    "training_regions":final_train,
    "validation_strategy":"Spatial-blocked H3 validation within the selected training geography; 3–4 folds depending on available H3 support.",
    "experiment_a_test_regions":final_a,
    "experiment_b_test_regions":final_b,
    "spatial_buffer_km":None if not np.isfinite(empirical_buffer) else float(empirical_buffer),
    "h3_leakage_rule":"No exact H3 overlap; remove training H3 cells within the empirically derived test buffer; inspect H3 neighbors/parent-child relationships before using raw H3 IDs.",
    "temporal_leakage_rule":"Historical/persistence features must be causal and recomputed inside each train/validation fold; no future information or target-derived encodings.",
    "selection_method":"India-wide candidate generation + hard support/leakage screening + equal-weight, weighted, Pareto and six-scenario sensitivity analysis.",
    "confidence":"DATA-DEPENDENT",
    "major_limitations":[]
}
if EXPLICIT_LABEL is None: design["major_limitations"].append("Observed taxonomy labels were not detected; non-gas class support is DATA UNAVAILABLE.")
if not np.isfinite(empirical_buffer): design["major_limitations"].append("Spatial autocorrelation range NOT COMPUTED.")
if wri is None: design["major_limitations"].append("WRI coverage DATA UNAVAILABLE.")
if env_pts.wc_class.notna().sum()==0: design["major_limitations"].append("ESA WorldCover sampling NOT COMPUTED.")
if len(osm_support)!=36: design["major_limitations"].append("OSM coverage incomplete.")

(WORK/"final_geographic_split.json").write_text(json.dumps(design,indent=2))
display(top5[["train_states","test_states","mean_rank","A_score","B_score","eco_novelty","feature_shift_score","spatial_separation_score","deployment_representativeness"]])

## Phase 17 — final report

In [ ]:
report=[]
report.append("# PS26162 Geographic Selection Report")
report.append("")
report.append(f"- FIRMS observations analysed: {len(joined):,}")
report.append(f"- India ADM1 units analysed: {india_states.state_ut.nunique()}")
report.append(f"- Unique H3 cells: {joined[hcol].nunique():,}" if hcol else "- Unique H3 cells: DATA UNAVAILABLE")
report.append("")
report.append("## Final design")
report.append(f"- TRAIN: {', '.join(final_train)}")
report.append("- VALIDATION: spatial-blocked H3 validation inside TRAIN")
report.append(f"- EXPERIMENT A: {', '.join(final_a)}")
report.append(f"- EXPERIMENT B: {', '.join(final_b)}")
report.append(f"- Spatial buffer: {empirical_buffer:.2f} km" if np.isfinite(empirical_buffer) else "- Spatial buffer: NOT COMPUTED")
report.append("")
report.append("## Evidence")
report.append(f"- Experiment A score: {best_a.A_score:.4f}")
report.append(f"- Experiment B score: {best_b.B_score:.4f}")
report.append(f"- A ecological novelty percentile: {best_a.eco_novelty:.4f}")
report.append(f"- B ecological novelty percentile: {best_b.eco_novelty:.4f}")
report.append(f"- A feature-space shift percentile: {best_a.feature_shift_score:.4f}")
report.append(f"- B feature-space shift percentile: {best_b.feature_shift_score:.4f}")
report.append("")
report.append("## Limitations")
for x in design["major_limitations"]: report.append(f"- {x}")
report.append("")
report.append("## Top five")
for i,(_,r) in enumerate(top5.iterrows(),1):
    report.append(f"{i}. TRAIN={r.train_states} | TEST={r.test_states} | mean sensitivity rank={r.mean_rank:.2f}")
(WORK/"geographic_selection_report.md").write_text("\n".join(report))

print("\n".join(report))

## Final decision output

In [ ]:
print("="*60)
print("FINAL GEOGRAPHIC EXPERIMENT DESIGN")
print("="*60)
print("TRAIN:", final_train)
print("VALIDATION: spatial-blocked H3 validation within TRAIN")
print("EXPERIMENT A:", final_a)
print("EXPERIMENT B:", final_b)
print("SPATIAL BUFFER:", f"{empirical_buffer:.2f} km" if np.isfinite(empirical_buffer) else "NOT COMPUTED")
print("H3 LEAKAGE RULE:", design["h3_leakage_rule"])
print("TEMPORAL LEAKAGE RULE:", design["temporal_leakage_rule"])
print("WHY THIS WON: empirical India-wide screening, hard support/leakage constraints, multi-objective ranking and sensitivity analysis.")
print("TOP ALTERNATIVE:", json.loads(top5.iloc[1].test_states) if len(top5)>1 else "NOT COMPUTED")
print("FOREST NOVELTY VERDICT: see Phase 14 quantitative distance.")
print("CONFIDENCE:", design["confidence"])
print("="*60)